# Part 1 — Descriptive Statistics
### Data Mondays, Week 4 — Amani Insurance claims case study

**Deliverable:** calculate and interpret **mean, median, mode, and standard deviation**.

We deliberately do a *quick, minimal* clean first — just enough to make the claim
amount column numeric — because descriptive stats on totally raw, uncleaned data
would be meaningless. The **full** missing-value/outlier treatment gets its own
notebook (Part 3) — we're not solving that problem here.

Run the cells in order, from top to bottom (Shift + Enter).

In [ ]:
import numpy as np
import pandas as pd

FILE_PATH = "insurance_claims_messy.csv"

## Section 1 — First look at the raw data

Before touching anything, just load the CSV and see what we're working with:
how many rows/columns, what the first few rows look like, and what data type
pandas assigned to each column.

In [ ]:
df = pd.read_csv(FILE_PATH)

print(f"Shape: {df.shape[0]} rows, {df.shape[1]} columns")
df.head(5)

In [ ]:
# Check the dtype pandas assigned each column.
# claim_amount_kes should be a number, but it isn't yet -- it's "object"
# (pandas' name for text), because the raw CSV has commas in numbers
# like "142,600", which pandas can't parse as a number on its own.
df.dtypes

## Section 2 — Minimum cleaning to make `claim_amount_kes` numeric

Two problems to fix before this column is usable:
1. Thousands-commas (`"142,600"`) need stripping before conversion.
2. Some cells are blank or contain garbage — `pd.to_numeric(..., errors="coerce")`
   turns anything it can't parse into `NaN` (pandas' "missing value" marker)
   instead of crashing.

In [ ]:
# .astype(str)          -> make sure every value is treated as text first
# .str.replace(",", "")  -> remove thousands-commas, e.g. "142,600" -> "142600"
# .str.strip()            -> remove stray leading/trailing spaces
df["claim_amount_kes"] = (
    df["claim_amount_kes"]
    .astype(str)
    .str.replace(",", "", regex=False)
    .str.strip()
)

# errors="coerce": anything that still isn't a valid number (blanks, junk)
# becomes NaN instead of raising an error and stopping the whole script.
df["claim_amount_kes"] = pd.to_numeric(df["claim_amount_kes"], errors="coerce")

print("Dtype after cleaning:", df["claim_amount_kes"].dtype)
print(f"Missing claim_amount_kes values: {df['claim_amount_kes'].isna().sum()}")
print(f"Negative claim_amount_kes values: {(df['claim_amount_kes'] < 0).sum()}")

In [ ]:
# For THIS notebook only, filter out missing/negative amounts into a
# separate variable `amounts`, so the stats below aren't distorted by data
# we haven't properly fixed yet. Notice we do NOT overwrite df itself --
# that keeps df intact for cells further down that need the full data.
amounts = df.loc[df["claim_amount_kes"] > 0, "claim_amount_kes"]
print(f"Usable amounts for this section: {len(amounts)} of {len(df)} rows")

## Section 3 — Mean, median, mode, standard deviation

Four numbers, four different questions about the same column:

| Statistic | Question it answers |
|---|---|
| **Mean** | What's the arithmetic average claim size? |
| **Median** | What claim size sits exactly in the middle, if sorted? |
| **Mode** | What single claim amount shows up most often? |
| **Std deviation** | How spread out are claim sizes around the average? |

In [ ]:
mean_amount = amounts.mean()
median_amount = amounts.median()
mode_amount = amounts.mode().iloc[0]   # .mode() can return several tied values; we take the first
std_amount = amounts.std()

print(f"Mean claim amount:   KES {mean_amount:>12,.0f}")
print(f"Median claim amount: KES {median_amount:>12,.0f}")
print(f"Mode claim amount:   KES {mode_amount:>12,.0f}  (most frequently occurring value)")
print(f"Std deviation:       KES {std_amount:>12,.0f}")

**Reading these numbers:** mean is noticeably higher than median here. That gap is
the signature of a **right-skewed** distribution — most claims are modest, but a
handful of very large claims (fire, theft) pull the *average* up higher than what
a "typical" claim actually looks like. This is a very common pattern in insurance
claims data, which is exactly why it's best practice to report both mean *and*
median rather than just one.

### Same calculations in NumPy

Pandas' `.mean()`, `.median()`, `.std()` are actually built on top of NumPy — let's
confirm that by recomputing the same numbers directly with NumPy functions.

In [ ]:
values = amounts.to_numpy()  # convert the pandas Series to a plain NumPy array

print(f"np.mean(values)   = {np.mean(values):,.0f}")
print(f"np.median(values) = {np.median(values):,.0f}")
print(f"np.std(values)    = {np.std(values):,.0f}  <-- note: differs slightly from pandas!")

**Why does NumPy's std differ slightly from pandas'?** `np.std()` divides by `n`
by default (the *population* formula), while pandas' `.std()` divides by `n - 1`
(the *sample* formula). We're almost always looking at a **sample** of all
possible claims, not literally every claim that could ever happen, so the sample
version is the standard default for real data. Pass `ddof=1` to make NumPy match
pandas exactly:

In [ ]:
print(f"np.std(values, ddof=1) = {np.std(values, ddof=1):,.0f}  <-- now matches pandas")

## Section 4 — Same four numbers, broken down by claim type

A single set of stats for *all* claims hides a lot. Motor-accident claims and
property-fire claims are not remotely on the same scale — let's split by
`claim_type` and compute mean/median/std/count for each group separately.

In [ ]:
# Canonicalise claim_type first, so "motor accident" / "MOTOR-ACCIDENT" /
# "Motor-Accident" all get counted as the SAME group instead of three
# different ones. (Full categorical cleanup is Part 3's job -- this is
# just enough to make groupby() behave sensibly here.)
df["claim_type_clean"] = (
    df["claim_type"].astype(str).str.strip().str.lower().str.replace(" ", "-")
)

usable = df[df["claim_amount_kes"] > 0]

# groupby(...).agg(...) computes all four statistics per group in one call.
summary = usable.groupby("claim_type_clean")["claim_amount_kes"].agg(
    mean="mean", median="median", std="std", count="count"
).sort_values("mean", ascending=False)

summary.round(0)

**Reading this table:** property-fire and motor-theft have both the highest
averages *and* the widest gap between mean and median — those are the claim
types dragging the whole dataset's skew. A large standard deviation relative to
the mean also tells an underwriter that claim type's payouts are hard to predict
in advance, which matters directly for how much an insurer needs to hold in
reserve for it.

**Up next:** Part 2 turns these same numbers into pictures — histograms and
boxplots — so we can *see* the skew we just calculated.